# Boosted Tree Class-Weight Variants

The unweighted boosted tree validated well locally but scored poorly publicly, likely because it predicted too many `at-risk` labels on test. This notebook trains several class-weight variants and saves separate submissions so we can test which distribution generalizes better.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)
RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"

## Load Data

In [2]:
train_df = pd.read_csv("data/train_split_features_numeric.csv")
val_df = pd.read_csv("data/val_split_features_numeric.csv")
test_df = pd.read_csv("data/test_features_numeric.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]
X_train = train_df[feature_cols].astype("float32")
X_val = val_df[feature_cols].astype("float32")
X_test = test_df[feature_cols].astype("float32")
y_train_raw = train_df[TARGET_COL]
y_val_raw = val_df[TARGET_COL]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_
class_id_by_name = {name: int(label_encoder.transform([name])[0]) for name in class_names}

print("class mapping:", class_id_by_name)
print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

class mapping: {'at-risk': 0, 'fit': 1, 'unhealthy': 2}
train: (552070, 73) val: (138018, 73) test: (295753, 72)


## Variant Configs

In [3]:
def class_weight_from_names(weights_by_name):
    if weights_by_name is None:
        return None
    return {class_id_by_name[name]: weight for name, weight in weights_by_name.items()}

variant_configs = [
    {
        "name": "unweighted",
        "weights_by_name": None,
    },
    {
        "name": "balanced",
        "weights_by_name": "balanced",
    },
    {
        "name": "mild_minority",
        "weights_by_name": {"at-risk": 0.75, "fit": 2.00, "unhealthy": 1.80},
    },
    {
        "name": "medium_minority",
        "weights_by_name": {"at-risk": 0.60, "fit": 2.80, "unhealthy": 2.30},
    },
    {
        "name": "nn_like_minority",
        "weights_by_name": {"at-risk": 0.50, "fit": 3.50, "unhealthy": 2.80},
    },
]

variant_configs

[{'name': 'unweighted', 'weights_by_name': None},
 {'name': 'balanced', 'weights_by_name': 'balanced'},
 {'name': 'mild_minority',
  'weights_by_name': {'at-risk': 0.75, 'fit': 2.0, 'unhealthy': 1.8}},
 {'name': 'medium_minority',
  'weights_by_name': {'at-risk': 0.6, 'fit': 2.8, 'unhealthy': 2.3}},
 {'name': 'nn_like_minority',
  'weights_by_name': {'at-risk': 0.5, 'fit': 3.5, 'unhealthy': 2.8}}]

## Train Validation Variants

In [4]:
def make_model(class_weight, max_iter=500, early_stopping=True):
    return HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.06,
        max_iter=max_iter,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        early_stopping=early_stopping,
        validation_fraction=0.15 if early_stopping else None,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
        class_weight=class_weight,
        verbose=0,
    )

validation_rows = []
validation_models = {}

for config in variant_configs:
    if config["weights_by_name"] == "balanced":
        class_weight = "balanced"
    else:
        class_weight = class_weight_from_names(config["weights_by_name"])

    print("Training", config["name"], "class_weight=", class_weight)
    model = make_model(class_weight=class_weight)
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    pred_labels = label_encoder.inverse_transform(pred)
    pred_distribution = pd.Series(pred_labels).value_counts(normalize=True).mul(100).to_dict()

    validation_rows.append({
        "variant": config["name"],
        "accuracy": accuracy_score(y_val_raw, pred_labels),
        "macro_f1": f1_score(y_val_raw, pred_labels, average="macro"),
        "weighted_f1": f1_score(y_val_raw, pred_labels, average="weighted"),
        "n_iter": model.n_iter_,
        "pred_at_risk_pct": pred_distribution.get("at-risk", 0),
        "pred_fit_pct": pred_distribution.get("fit", 0),
        "pred_unhealthy_pct": pred_distribution.get("unhealthy", 0),
    })
    validation_models[config["name"]] = model

validation_report = pd.DataFrame(validation_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
validation_report

Training unweighted class_weight= None


Training balanced class_weight= balanced


Training mild_minority class_weight= {0: 0.75, 1: 2.0, 2: 1.8}


Training medium_minority class_weight= {0: 0.6, 1: 2.8, 2: 2.3}


Training nn_like_minority class_weight= {0: 0.5, 1: 3.5, 2: 2.8}


,variant,accuracy,macro_f1,weighted_f1,n_iter,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,unweighted,0.966171,0.905995,0.964765,136,88.335579,4.958049,6.706372
1,mild_minority,0.962309,0.900574,0.961728,215,86.863308,5.273225,7.863467
2,medium_minority,0.957592,0.892009,0.957593,157,85.902563,5.489864,8.607573
3,nn_like_minority,0.947000,0.870886,0.948167,175,84.299874,6.210784,9.489342
4,balanced,0.882624,0.766829,0.894358,123,76.074860,10.278369,13.646771


## Save Submissions For Each Variant

Each final model is trained on train + validation using its validation-selected iteration count. The real test file is only predicted, never used for fitting.

In [5]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_df[feature_cols].astype("float32")
y_full = label_encoder.transform(full_df[TARGET_COL])

submission_rows = []

for config in variant_configs:
    variant_name = config["name"]
    validation_model = validation_models[variant_name]
    final_max_iter = int(validation_model.n_iter_)

    if config["weights_by_name"] == "balanced":
        class_weight = "balanced"
    else:
        class_weight = class_weight_from_names(config["weights_by_name"])

    print("Final training", variant_name, "max_iter=", final_max_iter, "class_weight=", class_weight)
    final_model = make_model(class_weight=class_weight, max_iter=final_max_iter, early_stopping=False)
    final_model.fit(X_full, y_full)

    test_pred = final_model.predict(X_test)
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_labels
    output_path = f"data/submission_boosted_{variant_name}.csv"
    submission.to_csv(output_path, index=False)

    dist = submission[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    submission_rows.append({
        "variant": variant_name,
        "path": output_path,
        "test_at_risk_pct": dist.get("at-risk", 0),
        "test_fit_pct": dist.get("fit", 0),
        "test_unhealthy_pct": dist.get("unhealthy", 0),
    })

submission_report = pd.DataFrame(submission_rows)
submission_report

Final training unweighted max_iter= 136 class_weight= None


Final training balanced max_iter= 123 class_weight= balanced


Final training mild_minority max_iter= 215 class_weight= {0: 0.75, 1: 2.0, 2: 1.8}


Final training medium_minority max_iter= 157 class_weight= {0: 0.6, 1: 2.8, 2: 2.3}


Final training nn_like_minority max_iter= 175 class_weight= {0: 0.5, 1: 3.5, 2: 2.8}


,variant,path,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,unweighted,data/submission_boosted_unweighted.csv,88.491410,4.938581,6.570009
1,balanced,data/submission_boosted_balanced.csv,75.610391,10.526351,13.863258
2,mild_minority,data/submission_boosted_mild_minority.csv,86.956683,5.257766,7.785551
3,medium_minority,data/submission_boosted_medium_minority.csv,85.969373,5.510003,8.520624
4,nn_like_minority,data/submission_boosted_nn_like_minority.csv,84.106670,6.288355,9.604974


## Reports

In [6]:
validation_report

,variant,accuracy,macro_f1,weighted_f1,n_iter,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,unweighted,0.966171,0.905995,0.964765,136,88.335579,4.958049,6.706372
1,mild_minority,0.962309,0.900574,0.961728,215,86.863308,5.273225,7.863467
2,medium_minority,0.957592,0.892009,0.957593,157,85.902563,5.489864,8.607573
3,nn_like_minority,0.947000,0.870886,0.948167,175,84.299874,6.210784,9.489342
4,balanced,0.882624,0.766829,0.894358,123,76.074860,10.278369,13.646771


In [7]:
submission_report

,variant,path,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,unweighted,data/submission_boosted_unweighted.csv,88.491410,4.938581,6.570009
1,balanced,data/submission_boosted_balanced.csv,75.610391,10.526351,13.863258
2,mild_minority,data/submission_boosted_mild_minority.csv,86.956683,5.257766,7.785551
3,medium_minority,data/submission_boosted_medium_minority.csv,85.969373,5.510003,8.520624
4,nn_like_minority,data/submission_boosted_nn_like_minority.csv,84.106670,6.288355,9.604974
